---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Learning a Linear Model

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)



# 🪄 1. Learning a Linear Model

Much of the power of data comes from its ability to help us predict unknown (e.g., future) quantities.

Let's build a simple model to predict the mpg of cars from the other information we have about them. Remember: mpg means miles per gallon.

The packages below are already included in the course environment. If needed, run `uv sync` in your terminal in the `wa` folder, then select the course's `.venv` Python environment as your notebook kernel.

## 1.1 Getting and cleaning our data

We'll use the SAME [Auto MPG dataset from UCI](https://archive.ics.uci.edu/dataset/9/auto+mpg) as in the pandas notebook. A copy is already in our `data` folder, so we don't need to download it again. Run the cells in order; later cells use things we created earlier.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LinearRegression

# notebooks can start in wa, scripts, or homework; let's find the course folder
course_folder = Path.cwd().resolve()
if course_folder.name in {'scripts', 'homework'}:
    course_folder = course_folder.parent
if not (course_folder / 'pyproject.toml').is_file():
    raise RuntimeError('Open the wa course folder in VS Code, then restart this notebook.')

data_folder = course_folder / 'data'

# define the names of the columns in our data
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
                'model_year', 'origin', 'car_name']

# this file separates values with spaces, rather than commas
mpg_df = pd.read_csv(data_folder / 'auto-mpg.data-original',
                     sep=r"\s+",
                     header=None,
                     names=column_names)

# display the first 5 rows
mpg_df.head(5)

In [ ]:
mpg_df.describe()

First, let's separate our **X** from our **y**.

- **X** contains the predictors (also called features): the information we give the model.
- **y** is the target: the number we want the model to predict.

For our cars, X will contain things like weight and horsepower, while y will be mpg.

The model we're using needs values for these columns. For now, we'll leave out cars with a missing predictor or target. This is a simple starting point, not a rule that we should delete every row with ANY missing value!

In [ ]:
# these are the columns we'll use to make predictions
predictors = ['weight', 'acceleration', 'horsepower', 'cylinders', 'displacement']

# and this is the column we want to predict
target = 'mpg'

# only drop rows missing something our model actually needs
cleaned_df = mpg_df.dropna(subset=predictors + [target])

print('Cars before cleaning:', len(mpg_df))
print('Cars we can use:', len(cleaned_df))

In [ ]:
cleaned_df[predictors].head()

In [ ]:
cleaned_df[target].head()

## 1.2 Linear regression

Let's start with something simple: a straight line through some points.

With one predictor, linear regression finds a line that represents the relationship between that predictor and our target. The line won't go through every point, and that's fine! The question is what we mean by the "best" line.

We'll use the **least squares criterion** to decide. More on this in a minute.

### Example

Consider some data on hours studied and exam scores. We'll use a small made-up example to make it easy to see what's happening.

In [ ]:
# sample data: hours studied vs. exam scores
data = {'Hours_Studied': [1, 2, 3, 4, 5, 6, 7, 8],
        'Exam_Score': [50, 55, 62, 65, 70, 75, 78, 85]}
df = pd.DataFrame(data)

# plotting the data
plt.scatter(df['Hours_Studied'], df['Exam_Score'], color='blue', label='Data Points')
plt.title('Hours Studied vs. Exam Score')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.legend()
plt.show()

In the plot above, the points seem to follow a linear trend: students who studied more hours have higher scores in our example.

### Fitting a linear regression model

When we say "fitting" a model, we mean finding its coefficients from the data.

Mathematically, our model is:

- $y=mx+c$ with one predictor $x$, where $m$ is the slope and $c$ is the intercept.
- $y=m_1x_1+m_2x_2+...+m_nx_n+c$ with multiple predictors. Each predictor gets its own coefficient.

For the study example, fitting means finding the slope and intercept of the line that best represents the relationship between hours studied and exam scores.

### What does *best* mean?

The code below has some candidate lines. Which one do you think is the best fit?

In [ ]:
# we already created df above; let's use it to draw some candidate lines
lines = {
    'Line 1': [48 + 5*x for x in df['Hours_Studied']],
    'Line 2': [45 + 6*x for x in df['Hours_Studied']],
    'Line 3': [50 + 4.5*x for x in df['Hours_Studied']],
    'Line 4': [70 + 0.2*x for x in df['Hours_Studied']]
}

# plotting the data and the potential fit lines
plt.scatter(df['Hours_Studied'], df['Exam_Score'], color='blue', label='Data Points')

for line, values in lines.items():
    plt.plot(df['Hours_Studied'], values, label=line)

plt.title('Potential Fit Lines: Hours Studied vs. Exam Score')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.legend()
plt.show()

## Least Squares Criterion as an Objective Function

The fundamental idea behind the method of least squares is quite intuitive: Find the line (or curve) that minimizes the sum of the squared differences (residuals) between the observed values and the values that our model predicts.

### A. Why Squares?
Before diving into the math, it's worth noting why we consider the squared differences:
1. **Positivity**: Squaring ensures that negative and positive differences don't cancel each other out.
2. **Penalty**: Larger deviations are given more weight, meaning we're penalizing large deviations more than smaller ones.
3. **Differentiability**: In calculus, squared functions are differentiable, which is essential for optimization techniques.

### B. Mathematical Representation

Suppose you have a set of data points $\{(x_1, y_1), (x_2, y_2), ..., (x_n, y_n)\} $. 
Now, let's assume that the relationship between $x$ and $y$ is linear and can be represented as $ y = \beta_0 + \beta_1 x $, where
- $ \beta_0 $ is the y-intercept.
- $ \beta_1 $ is the slope of the line.

For each data point $ x_i $, your model would then predict a y-value equal to $\hat{y_i} = \beta_0 + \beta_1 x_i $.

The residual (difference between observed and predicted) for this data point is $ e_i = y_i - \hat{y_i} $.

The goal of the least squares method is to find values for $ \beta_0 $ and $ \beta_1 $ that minimize the sum of the squared residuals:

$ Q(\beta_0, \beta_1) = \sum_{i=1}^{n} e_i^2 = \sum_{i=1}^{n} (y_i - \beta_0 - \beta_1 x_i)^2 $

#### C. Optimization (Optional)

To find the values of $\beta_0$ and $\beta_1$ that minimize $Q$, we set its partial derivatives with respect to $\beta_0$ and $\beta_1$ to zero. The resulting system of equations can be solved to get:
- $\beta_1 = \frac{n(\sum xy) - (\sum x)(\sum y)}{n(\sum x^2) - (\sum x)^2} $
- $\beta_0 = \frac{\sum y - \beta_1 \sum x}{n}$
Where:
- $ n $ is the number of data points in our dataset.
- The sums $\sum x$, $\sum y$, $\sum xy$, and $\sum x^2$ are taken over all the data points.

In [ ]:
# create a model; it hasn't learned anything yet!
study_model = LinearRegression()

# double brackets keep X as a table, even though it has only one column
# y is a single column: the scores we want to predict
X = df[['Hours_Studied']]
y = df['Exam_Score']

# fit() finds the slope and intercept using X and the known scores in y
study_model.fit(X, y)

# predict() uses the fitted model to calculate scores from the hours in X
df['Predicted_Score'] = study_model.predict(X)

# plotting the data, candidate lines, and best fit line
plt.scatter(df['Hours_Studied'], df['Exam_Score'], color='blue', label='Data Points')

for line, values in lines.items():
    plt.plot(df['Hours_Studied'], values, label=line)

plt.plot(df['Hours_Studied'], df['Predicted_Score'],
         color='red', linestyle='--', label='Best Fit Line')

plt.title('Potential Fit Lines and Best Fit Line: Hours Studied vs. Exam Score')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.legend()
plt.show()

One advantage of fitting a linear regression model is that we can inspect its coefficients.

Here, we have only one predictor. Its coefficient is the **slope**: how much the predicted exam score changes for one additional hour of study. The **intercept** is the value of our fitted line when hours studied is zero.

Let's look at both.

In [ ]:
# coef_ contains one coefficient per predictor; here we only have one
print('Slope:', study_model.coef_[0])
print('Intercept:', study_model.intercept_)

For our example, each additional hour of study is associated with about a 4.8-point increase in the exam score.

That is NOT necessarily a causal relationship! A fitted line describes a relationship in the data; it doesn't prove that studying one more hour causes exactly that increase.

The intercept is about 45.9. That's what our line predicts at zero hours, but our example only has students who studied between 1 and 8 hours. Don't treat a prediction outside the data we've seen as a guarantee!

## 1.3 Back to cars

Now let's do the same thing with our cars. The main difference is that we have FIVE predictors instead of just one.

Each coefficient tells us how the model's predicted mpg changes for a one-unit increase in that predictor, **holding the other predictors fixed**. For example, the weight coefficient compares predictions when weight changes but horsepower, cylinders, displacement, and acceleration stay the same.

Again, these are relationships learned from the data, not proof of cause and effect. Also, the predictors use different units, so we can't decide which one is most important just by comparing the sizes of their coefficients.

In [ ]:
# build a separate model for cars
car_model = LinearRegression()

X_cars = cleaned_df[predictors]
y_cars = cleaned_df[target]

# fit() learns one coefficient per predictor, plus an intercept
car_model.fit(X_cars, y_cars)

print('Intercept:', car_model.intercept_)

# zip pairs each predictor name with its coefficient, in the same order
pd.DataFrame([dict(zip(predictors, car_model.coef_))])

We fitted our linear regression. Let's now see what it predicts!

Each dot in the next plot is one car: its actual mpg is on the horizontal axis, and our predicted mpg is on the vertical axis.

But keep in mind: these are the SAME cars we used to fit the model. We've already shown it their mpg! Looking good here doesn't tell us how well we'll predict cars we haven't seen. We'll cover proper testing in the next lecture.

The dashed diagonal shows **perfect prediction**: actual mpg equals predicted mpg. The closer a dot is to this line, the closer our prediction was! A dot below the line means we predicted too low; a dot above it means we predicted too high.

In [ ]:
# predict() uses the coefficients we learned; it does not fit the model again
preds = car_model.predict(X_cars)

# add predictions to a new dataframe, keeping cleaned_df unchanged
predictions_df = cleaned_df.assign(predictions=preds)
predictions_df.head(10)

In [ ]:
# compare predictions with actual mpg on the SAME scale
ax = predictions_df.plot(kind='scatter',
                         x='mpg',
                         y='predictions',
                         c='forestgreen',
                         alpha=0.7,
                         figsize=(6, 6),
                         xlim=(0, 50),
                         ylim=(0, 50))

# this is a reference line, NOT the fitted regression line
ax.plot([0, 50], [0, 50], color='black', linestyle='--',
        label='Perfect prediction')

ax.set_xlabel('Actual MPG')
ax.set_ylabel('Predicted MPG')
ax.set_title(f'Actual vs. Predicted MPG\nTraining cars (n={len(predictions_df)})')

# equal spacing on both axes makes the diagonal a 45-degree line
ax.set_aspect('equal', adjustable='box')
ax.legend()
plt.tight_layout()
plt.show()

### Question: Why didn't we plot the best-fitting line -- like before?

With one predictor, we can draw the model as a line using two axes: the predictor and the target.

With two predictors, the model is a plane in three dimensions. With more predictors, we need still more dimensions to show the whole model. Here, we have five predictors, so one ordinary 2-dimensional plot can't show all of them at once.

Instead, we plotted actual mpg against predicted mpg. This lets us compare the predictions without trying to draw every predictor on a separate axis. The dashed diagonal is just a guide for checking predictions, not our fitted model!